# Forward Pass

A **forward pass** sends data through the network to get a prediction.

```mermaid
flowchart LR
    I[Input] --> L1[Linear Layer]
    L1 --> A[Activation]
    A --> L2[Linear Layer]
    L2 --> O[Output / Prediction]
```

The output depends on the task:

- Binary classification: one probability
- Multi-class classification: one probability per class
- Regression: one numerical value

In [1]:
import torch
from torch import nn

torch.manual_seed(42)

# 5 animals, 6 features per animal
input_data = torch.tensor([
    [-0.4421,  1.5207,  2.0607, -0.3647,  0.4691,  0.0940],
    [-0.9155, -0.0475, -1.3645,  0.6336, -1.9520, -0.3398],
    [ 0.7406,  1.6763, -0.8511,  0.2432,  0.1123, -0.0633],
    [-1.6630, -0.0718, -0.1285,  0.5396, -0.0288, -0.8622],
    [-0.7413,  1.7920, -0.0883, -0.6685,  0.4745, -0.4245],
])

print(input_data.shape)

torch.Size([5, 6])


## Binary Classification

Example: `mammal` or `not mammal`.

```mermaid
flowchart LR
    I[6 features] --> H[Linear: 6 to 4]
    H --> O[Linear: 4 to 1]
    O --> S[Sigmoid]
    S --> P[1 probability]
```

`Sigmoid` keeps the output between `0` and `1`. Use `0.5` as the threshold.

In [2]:
binary_model = nn.Sequential(
    nn.Linear(6, 4),
    nn.Linear(4, 1),
    nn.Sigmoid(),
)

binary_probabilities = binary_model(input_data)
binary_labels = (binary_probabilities >= 0.5).int()

print(f"Output shape: {binary_probabilities.shape}")
print(f"Probabilities:\n{binary_probabilities}")
print(f"Labels:\n{binary_labels}")

Output shape: torch.Size([5, 1])
Probabilities:
tensor([[0.5258],
        [0.5784],
        [0.6498],
        [0.5418],
        [0.5037]], grad_fn=<SigmoidBackward0>)
Labels:
tensor([[1],
        [1],
        [1],
        [1],
        [1]], dtype=torch.int32)


## Multi-class Classification

Example: `mammal`, `bird`, or `reptile`.

```mermaid
flowchart LR
    I[6 features] --> H[Linear: 6 to 4]
    H --> O[Linear: 4 to 3]
    O --> S[Softmax]
    S --> P[3 probabilities]
```

`dim=-1` applies softmax to the last dimension: the `3` class scores in shape `[5, 3]`. Each row will sum to `1`.

### What Does `dim=-1` Mean?

A dimension is a direction in the tensor. For output shape `[5, 3]`:

```text
                         dim=-1 / dim=1 -> classes
                    mammal      bird      reptile
animal 1          [ score,      score,      score ]
animal 2          [ score,      score,      score ]
animal 3          [ score,      score,      score ]
animal 4          [ score,      score,      score ]
animal 5          [ score,      score,      score ]
     |
     v
   dim=0
  animals
```

- `dim=0` moves down through the five animals.
- `dim=1` moves across the three classes.
- `dim=-1` means the last dimension, so here it is the same as `dim=1`.

Softmax should compare the three classes **inside each animal's row**. That is why we use `dim=-1`.

```mermaid
flowchart LR
    S[One animal's 3 scores] --> SM[Softmax along dim=-1]
    SM --> P[3 probabilities that sum to 1]
```

In [3]:
example_scores = torch.tensor([
    [2.0, 1.0, 0.5],  # animal 1: 3 class scores
    [0.2, 1.5, 3.0],  # animal 2: 3 class scores
])

# dim=-1 converts each animal's 3 scores into probabilities
probabilities_by_animal = torch.softmax(example_scores, dim=-1)

print(f"Scores shape: {example_scores.shape}")
print(f"Probabilities by animal:\n{probabilities_by_animal}")
print(f"Each animal's row sums to: {probabilities_by_animal.sum(dim=-1)}")

# dim=0 would compare different animals with each other instead
probabilities_by_column = torch.softmax(example_scores, dim=0)
print(f"\nIf dim=0 is used:\n{probabilities_by_column}")
print(f"Each class column sums to: {probabilities_by_column.sum(dim=0)}")

Scores shape: torch.Size([2, 3])
Probabilities by animal:
tensor([[0.6285, 0.2312, 0.1402],
        [0.0474, 0.1738, 0.7789]])
Each animal's row sums to: tensor([1.0000, 1.0000])

If dim=0 is used:
tensor([[0.8581, 0.3775, 0.0759],
        [0.1419, 0.6225, 0.9241]])
Each class column sums to: tensor([1.0000, 1.0000, 1.0000])


`-1` always means **the final dimension**, even when the tensor has more dimensions:

```text
shape [5, 3]      -> dim=-1 is the 3 classes
shape [5, 10, 3]  -> dim=-1 is still the 3 classes
```

This makes `dim=-1` more flexible than writing a fixed dimension number.

In [4]:
class_names = ["mammal", "bird", "reptile"]

multiclass_model = nn.Sequential(
    nn.Linear(6, 4),
    nn.Linear(4, len(class_names)),
    nn.Softmax(dim=-1),
)

class_probabilities = multiclass_model(input_data)
predicted_indices = class_probabilities.argmax(dim=-1)
predicted_labels = [class_names[index] for index in predicted_indices]

print(f"Output shape: {class_probabilities.shape}")
print(f"Probabilities:\n{class_probabilities}")
print(f"Row sums: {class_probabilities.sum(dim=-1)}")
print(f"Predictions: {predicted_labels}")

Output shape: torch.Size([5, 3])
Probabilities:
tensor([[0.3005, 0.2892, 0.4103],
        [0.3583, 0.3057, 0.3359],
        [0.2990, 0.4362, 0.2648],
        [0.3046, 0.2586, 0.4369],
        [0.3007, 0.3745, 0.3248]], grad_fn=<SoftmaxBackward0>)
Row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)
Predictions: ['reptile', 'mammal', 'bird', 'reptile', 'bird']


## Regression

Example: predict each animal's weight.

```mermaid
flowchart LR
    I[6 features] --> H[Linear: 6 to 4]
    H --> O[Linear: 4 to 1]
    O --> V[1 numerical value]
```

There is no final activation function. The output shape is `[5, 1]`: one value for each animal.

In [ ]:
regression_model = nn.Sequential(
    nn.Linear(6, 4),
    nn.Linear(4, 1),
)

predicted_weights = regression_model(input_data)

print(f"Output shape: {predicted_weights.shape}")
print(f"Predicted weights:\n{predicted_weights}")

> These predictions are random for now. Training updates the weights and biases so the outputs become useful.